# Linguistic Features Pipeline

This notebook extracts linguistic features for the gaze-guided text generation study:
- **Word frequency**: SUBTLEX-US Zipf scores (lemma-first, then surface)
- **Word length**: Alphabetic character count
- **Dependency distance**: Linear distance to syntactic head (Gibson 2000 locality theory)
- **Syntactic tree depth**: Distance from token to root
- **Integration cost**: Locality-based processing difficulty metric

Features are aligned to AOI tokens and merged with TRT data for mixed-effects modeling.

In [1]:
# Install dependencies with version pins for reproducibility
%pip install -q "spacy==3.7.4" "polars==1.6.0" "pyarrow==15.0.2" "wordfreq==3.1.1"

# Download spaCy model if not present
import subprocess
import sys
try:
    import en_core_web_sm
except ImportError:
    print("Downloading spaCy English model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

Note: you may need to restart the kernel to use updated packages.


C:\Python312\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [2]:
from pathlib import Path
import json
import re
import polars as pl
import spacy

# Paths
BASE = Path("data-clean")
RAW = BASE / "raw"
TEXTS_DIR = RAW / "texts"
STIMULI_DIR = RAW / "stimuli"
PROC = BASE / "processed"
RESOURCES = BASE / "resources"

# Ensure output directory exists
PROC.mkdir(parents=True, exist_ok=True)

# File paths
TRT_PARQUET = PROC / "trt_by_word.parquet"
TRT_CSV = PROC / "trt_by_word.csv"
SUBTLEX_PATH = RESOURCES / "SUBTLEX-US.csv"

print(f"Base directory: {BASE.absolute()}")
print(f"SUBTLEX exists: {SUBTLEX_PATH.exists()}")
print(f"TRT data exists: {TRT_PARQUET.exists() or TRT_CSV.exists()}")

Base directory: c:\a\project\data-clean
SUBTLEX exists: True
TRT data exists: True


In [3]:
# Helper functions
def is_practice(name: str) -> bool:
    return "practice" in name.lower()

def extract_condition(name: str) -> str:
    lname = name.lower()
    if "neg" in lname: return "neg"
    if "pos" in lname: return "pos"  
    if "zero" in lname: return "zero"
    return "unknown"

# Text normalization for alignment: keep apostrophes & hyphens
_punct_re = re.compile(r"[^\w'\-]+", flags=re.UNICODE)

def normalize_token(s: str) -> str:
    """Normalize token for alignment: lowercase; keep apostrophes & hyphens; strip other punct."""
    return _punct_re.sub("", s.lower())

def align_aoi_to_spacy_windowed(aoi_tokens: list[str], doc_tokens: list[str], max_window: int = 2) -> list[int | None]:
    """
    Greedy left-to-right alignment by normalized surface forms.
    Supports concatenating up to `max_window` spaCy tokens to match hyphenated/multiword AOIs.
    Also aligns punctuation-only AOIs to identical doc tokens.
    """
    mapping: list[int | None] = [None] * len(aoi_tokens)
    j = 0
    N = len(doc_tokens)

    for i, aoi_tok in enumerate(aoi_tokens):
        raw = aoi_tok.strip()
        tgt = normalize_token(aoi_tok)

        # Handle pure punctuation AOIs by literal match
        if tgt == "" and raw:
            while j < N and doc_tokens[j].strip() != raw:
                j += 1
            if j < N and doc_tokens[j].strip() == raw:
                mapping[i] = j
                j += 1
            continue

        if tgt == "":
            # empty after normalization; skip
            continue

        matched = False
        k = j
        while k < N and not matched:
            for w in range(1, max_window + 1):
                if k + w > N:
                    break
                window_norm = "".join(normalize_token(t) for t in doc_tokens[k:k + w])
                if window_norm == tgt:
                    mapping[i] = k
                    j = k + w
                    matched = True
                    break
            if not matched:
                k += 1

        if not matched:
            j = min(j + 1, N)

    return mapping

In [4]:
# Load TRT data to get stimulus list (excludes practice)
if TRT_PARQUET.exists():
    trt = pl.read_parquet(TRT_PARQUET)
else:
    trt = pl.read_csv(TRT_CSV)

# Filter out practice stimuli
trt = trt.filter(~pl.col("stimulus").str.contains("practice"))
stimuli_to_process = trt["stimulus"].unique().to_list()

print(f"Stimuli to process (from TRT): {len(stimuli_to_process)}")
print(f"Subjects: {trt['subject_id'].n_unique()}")
print(f"Conditions: {sorted(trt['condition'].unique().to_list())}")

Stimuli to process (from TRT): 152
Subjects: 12
Conditions: ['neg', 'pos', 'zero']


In [5]:
# Load AOI data for alignment
def load_aois(stimulus: str) -> pl.DataFrame | None:
    """Load AOI word-level data for a stimulus."""
    aoi_path = STIMULI_DIR / f"{stimulus}.word.csv"
    if not aoi_path.exists():
        return None
        
    aoi_df = pl.read_csv(aoi_path)
    required_cols = {"index", "content", "left", "right", "top", "bottom"}
    missing_cols = required_cols - set(aoi_df.columns)
    
    if missing_cols:
        raise ValueError(f"AOI file missing columns {missing_cols} for {stimulus}")
        
    return aoi_df.sort("index")  # Ensure consistent token order

# Load AOIs for all stimuli
stimulus_aois: dict[str, pl.DataFrame] = {}
missing_aois = []

for stimulus in stimuli_to_process:
    if is_practice(stimulus):
        continue
        
    aoi_df = load_aois(stimulus)
    if aoi_df is not None:
        stimulus_aois[stimulus] = aoi_df
    else:
        missing_aois.append(stimulus)

print(f"Loaded AOIs: {len(stimulus_aois)} stimuli")
if missing_aois:
    print(f"Missing AOI files: {missing_aois}")

Loaded AOIs: 152 stimuli


In [6]:
# Text loading with fallbacks
def load_text(stimulus: str, aoi_df: pl.DataFrame) -> str:
    """Load stimulus text from JSON, TXT, or reconstruct from AOIs."""
    # Try JSON first
    json_path = TEXTS_DIR / f"{stimulus}.json"
    if json_path.exists():
        try:
            data = json.loads(json_path.read_text(encoding="utf-8"))
            # Handle common JSON structures
            if isinstance(data, dict):
                for key in ("text", "content", "body"):
                    if key in data and isinstance(data[key], str) and data[key].strip():
                        return data[key]
                # Fallback: first string value
                for value in data.values():
                    if isinstance(value, str) and value.strip():
                        return value
        except (json.JSONDecodeError, UnicodeDecodeError):
            pass
    
    # Try TXT file
    txt_path = TEXTS_DIR / f"{stimulus}.txt" 
    if txt_path.exists():
        try:
            return txt_path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            pass
    
    # Fallback: reconstruct from AOI content
    return " ".join(aoi_df["content"].to_list())

# Test text loading for first few stimuli
for i, (stimulus, aoi_df) in enumerate(list(stimulus_aois.items())[:3]):
    text = load_text(stimulus, aoi_df)
    print(f"{stimulus}: {len(text)} chars, first 100: {text[:100]!r}")
    if i >= 2:  # Only show first 3
        break

prize-neg.text.1: 725 chars, first 100: 'As she grew older, the feeling of coming in second place began to take a toll on her. She started to'
blackout-neg.text.2: 549 chars, first 100: 'open, just a crack. It was as if someone had been waiting for them, had been waiting for them to com'
delayed-neg.text.1: 516 chars, first 100: '"Mind if I join you?" it said. Sarah turned to see a man with a kind face and a warm smile. He was h'


In [7]:
# Initialize spaCy pipeline
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm", exclude=["ner"])  # Exclude NER for speed

# Ensure sentence segmentation
if not nlp.has_pipe("senter") and not nlp.has_pipe("parser"):
    nlp.add_pipe("sentencizer")

print(f"spaCy pipeline: {nlp.pipe_names}")

# Test parsing
test_text = "The quick brown fox jumps over the lazy dog."
doc = nlp(test_text)
print(f"Test parse: {len(doc)} tokens, {len(list(doc.sents))} sentences")

Loading spaCy model...


spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer']
Test parse: 10 tokens, 1 sentences


In [8]:
# Load SUBTLEX-US frequency data
from math import log10

def load_subtlex(path: Path) -> pl.DataFrame:
    """Load SUBTLEX data with robust column detection and scaling.
    Preference order:
    1) SUBTLWF (freq per million): Zipf = log10(SUBTLWF) + 3
    2) Any 'zipf' column: use as-is
    3) lg10WF: ONLY use as-is (no +3), because it's log10(raw frequency), not per million
       and adding +3 would inflate scales. This keeps it monotonic and comparable.
    """
    if not path.exists():
        raise FileNotFoundError(f"SUBTLEX file not found at {path}")

    df_raw = pl.read_csv(path, separator="\t", infer_schema_length=50000)
    cols_lc = {c.lower(): c for c in df_raw.columns}

    # word/form column
    word_col = None
    for candidate in ["word", "spelling", "lemma", "wordform"]:
        if candidate in cols_lc:
            word_col = cols_lc[candidate]
            break
    if not word_col:
        word_col = df_raw.columns[0]

    # 1) Prefer SUBTLWF if present (per million)
    subtlwf_col = None
    for k in ["subtlwf", "freqpm", "freq_per_million"]:
        if k in cols_lc:
            subtlwf_col = cols_lc[k]
            break

    if subtlwf_col:
        zipf_expr = (
            pl.when(pl.col(subtlwf_col).cast(pl.Float64) > 0)
            .then(pl.col(subtlwf_col).cast(pl.Float64).log10() + 3.0)
            .otherwise(None)
        )
    else:
        # 2) Any 'zipf' column
        zipf_candidates = [c for c in df_raw.columns if "zipf" in c.lower()]
        if zipf_candidates:
            base_col = zipf_candidates[0]
            zipf_expr = pl.col(base_col).cast(pl.Float64)
        else:
            # 3) lg10WF (log10 raw counts): use as-is to avoid artificial offset
            lg10_col = None
            for k in ["lg10wf", "log10wf", "lg10_wf"]:
                if k in cols_lc:
                    lg10_col = cols_lc[k]
                    break
            if lg10_col:
                zipf_expr = pl.col(lg10_col).cast(pl.Float64)
            else:
                raise ValueError("SUBTLEX file must have SUBTLWF, Zipf, or lg10WF column")

    result = (
        df_raw.select([
            pl.col(word_col).str.to_lowercase().alias("form"),
            zipf_expr.alias("zipf"),
        ])
        .filter(pl.col("form").str.len_chars() > 0)
        .filter(pl.col("zipf").is_not_null())
        .group_by("form").agg(pl.col("zipf").max())
    )
    return result

# Load SUBTLEX
print("Loading SUBTLEX-US...")
subtlex = load_subtlex(SUBTLEX_PATH)
print(f"SUBTLEX entries: {subtlex.height}")
print("Sample entries:")
print(subtlex.head().to_pandas())

Loading SUBTLEX-US...
SUBTLEX entries: 74286
Sample entries:


         form      zipf
0   springbok  1.301030
1      noires  1.301030
2    outraced  1.301030
3      deeply  4.159266
4  criticised  2.342423


In [9]:
# Dependency Locality Theory functions (Gibson 2000)

def token_depth(token) -> int:
    """Calculate syntactic tree depth: distance from token to root."""
    depth = 0
    current = token
    while current.head != current:  # Until we reach root
        depth += 1
        current = current.head
        if depth > 50:  # Prevent infinite loops
            break
    return depth

def dependency_distance(token) -> int:
    """Calculate linear dependency distance: |position - head_position|."""
    return abs(token.i - token.head.i)

def integration_cost(token) -> float:
    """
    Calculate integration cost based on Gibson 2000 Dependency Locality Theory.
    
    Integration cost reflects processing difficulty due to:
    1. Linear distance to syntactic head
    2. Number of discourse referents between dependent and head
    
    Simplified metric: linear distance weighted by dependency type.
    """
    if token.head == token:  # Root has no integration cost
        return 0.0
    
    dist = dependency_distance(token)
    
    # Weight by dependency relation importance (simplified)
    # More important relations have higher integration costs
    relation_weights = {
        "nsubj": 1.0,     # Subject
        "dobj": 1.0,      # Direct object  
        "prep": 0.8,      # Prepositional
        "amod": 0.6,      # Adjectival modifier
        "advmod": 0.6,    # Adverbial modifier
        "det": 0.3,       # Determiner
        "aux": 0.3,       # Auxiliary
        "punct": 0.1,     # Punctuation
    }
    
    weight = relation_weights.get(token.dep_, 0.5)  # Default weight
    
    # Integration cost = distance * relation_weight
    # Add small penalty for very long distances
    cost = dist * weight
    if dist > 5:
        cost += (dist - 5) * 0.1  # Additional penalty for long dependencies
        
    return cost

def locality_features(token) -> dict:
    """Extract all locality-related features for a token."""
    return {
        "dep_dist": dependency_distance(token),
        "depth": token_depth(token),
        "integration_cost": integration_cost(token),
        "dep_label": token.dep_,
        "pos_tag": token.pos_,
    }

In [10]:
# Frequency lookup function
from wordfreq import zipf_frequency

def lookup_zipf(surface: str, lemma: str = "") -> float | None:
    """
    Look up Zipf frequency score.
    SUBTLEX-first, then wordfreq fallback. Candidates include lemma/surface,
    strip possessive 's, de-hyphenized form, and hyphen parts.
    """
    candidates: list[str] = []
    if lemma:
        candidates.append(lemma)
    if surface:
        candidates.append(surface)
        low = surface.lower()
        # strip possessive
        if low.endswith(("’s", "'s")) and len(surface) > 2:
            candidates.append(surface[:-2])
        # de-hyphenated and parts
        if "-" in surface:
            candidates.append(surface.replace("-", ""))
            candidates.extend([p for p in surface.split("-") if p])

    # Normalize and deduplicate
    norm = []
    seen = set()
    for c in candidates:
        n = normalize_token(c)
        if n and n not in seen:
            seen.add(n)
            norm.append(n)

    # 1) SUBTLEX lookup
    for n in norm:
        hit = subtlex.filter(pl.col("form") == n).select("zipf")
        if hit.height:
            return float(hit.item())

    # 2) wordfreq fallback
    for n in norm:
        z = zipf_frequency(n, "en")
        if z > 0:
            return float(z)

    return None  # Not found

# Test frequency lookup
test_words = ["the", "cat", "quickly", "xyzabc"]
for word in test_words:
    freq = lookup_zipf(word)
    print(f"{word}: {freq}")

the: 7.469073206544746
cat: 4.821709997298376
quickly: 4.751971574736327
xyzabc: None


In [11]:
# Main feature extraction
print("Extracting linguistic features...")

feature_tables = []
coverage_stats = []

for stimulus_idx, (stimulus, aoi_df) in enumerate(stimulus_aois.items()):
    print(f"Processing {stimulus_idx+1}/{len(stimulus_aois)}: {stimulus}")
    
    # Load and parse text
    text = load_text(stimulus, aoi_df)
    doc = nlp(text)
    
    # Get non-space tokens for alignment
    doc_tokens_all = [t for t in doc if not t.is_space]
    doc_tokens_text = [t.text for t in doc_tokens_all]
    
    # Get AOI tokens in order
    aoi_tokens = aoi_df.sort("index")["content"].to_list()
    aoi_indices = aoi_df.sort("index")["index"].to_list()
    
    # Align AOI tokens to spaCy tokens (windowed)
    alignment = align_aoi_to_spacy_windowed(aoi_tokens, doc_tokens_text, max_window=2)
    
    # Calculate coverage
    aligned_count = sum(1 for x in alignment if x is not None)
    coverage = aligned_count / max(1, len(aoi_tokens))
    coverage_stats.append((stimulus, coverage, len(aoi_tokens), aligned_count))
    
    # Extract features for each AOI token
    rows = []
    for aoi_idx, content, spacy_idx in zip(aoi_indices, aoi_tokens, alignment):
        if spacy_idx is None:
            # Unaligned token - compute basic features only
            rows.append({
                "stimulus": stimulus,
                "index": int(aoi_idx),
                "content": content,
                "word_len": sum(c.isalpha() for c in content),
                "freq_zipf": None,
                "dep_dist": None,
                "depth": None,
                "integration_cost": None,
                "dep_label": None,
                "pos_tag": None,
                "lemma": None,
                "sentence_id": None,
                "token_id_sent": None,
            })
            continue
        
        # Get spaCy token
        token = doc_tokens_all[spacy_idx]
        
        # Calculate features
        word_len = sum(c.isalpha() for c in content)
        lemma = token.lemma_ if hasattr(token, 'lemma_') else ""
        freq_zipf = lookup_zipf(content, lemma)
        
        # Locality features  
        locality = locality_features(token)
        
        # Sentence information
        sentence_id = None
        token_id_sent = None
        for sent_idx, sent in enumerate(doc.sents):
            if token.i >= sent.start and token.i < sent.end:
                sentence_id = sent_idx
                token_id_sent = token.i - sent.start
                break
        
        rows.append({
            "stimulus": stimulus,
            "index": int(aoi_idx),
            "content": content,
            "word_len": word_len,
            "freq_zipf": freq_zipf,
            "dep_dist": locality["dep_dist"],
            "depth": locality["depth"], 
            "integration_cost": locality["integration_cost"],
            "dep_label": locality["dep_label"],
            "pos_tag": locality["pos_tag"],
            "lemma": lemma,
            "sentence_id": sentence_id,
            "token_id_sent": token_id_sent,
        })
    
    # Convert to DataFrame
    stimulus_features = pl.from_records(rows)
    feature_tables.append(stimulus_features)

print("Feature extraction complete.")

Extracting linguistic features...
Processing 1/152: prize-neg.text.1
Processing 2/152: blackout-neg.text.2


Processing 3/152: delayed-neg.text.1


Processing 4/152: voicemail-neg.text.1
Processing 5/152: voicemail-pos.text.1
Processing 6/152: goldfish-pos.text.4


Processing 7/152: delayed-neg.text.4


Processing 8/152: breakfast-zero.difficulty
Processing 9/152: prize-pos.text.2
Processing 10/152: prize-zero.interest
Processing 11/152: voicemail-zero.text.0
Processing 12/152: goldfish-pos.text.1


...

Processing 121/152: prize-zero.naturalness
Processing 122/152: goldfish-neg.difficulty
Processing 123/152: blackout-zero.naturalness
Processing 124/152: delayed-zero.interest
Processing 125/152: blackout-pos.text.3
Processing 126/152: goldfish-pos.text.0
Processing 127/152: blackout-zero.text.2


Processing 128/152: breakfast-zero.text.2
Processing 129/152: prize-pos.text.3
Processing 130/152: voicemail-neg.difficulty
Processing 131/152: breakfast-zero.text.1
Processing 132/152: breakfast-zero.text.3


Processing 133/152: prize-pos.interest
Processing 134/152: prize-neg.interest
Processing 135/152: goldfish-neg.interest
Processing 136/152: breakfast-pos.interest
Processing 137/152: breakfast-neg.question
Processing 138/152: blackout-pos.difficulty
Processing 139/152: blackout-neg.difficulty
Processing 140/152: voicemail-neg.text.2
Processing 141/152: delayed-pos.naturalness
Processing 142/152: prize-zero.text.0


Processing 143/152: goldfish-zero.text.0
Processing 144/152: delayed-pos.text.1
Processing 145/152: voicemail-pos.difficulty
Processing 146/152: voicemail-zero.text.3
Processing 147/152: voicemail-neg.naturalness
Processing 148/152: blackout-pos.interest
Processing 149/152: goldfish-zero.text.1


Processing 150/152: delayed-neg.interest
Processing 151/152: prize-neg.naturalness
Processing 152/152: prize-zero.question
Feature extraction complete.


In [12]:
# Consolidate all features
features = pl.concat(feature_tables, how="vertical_relaxed")

# Cast to appropriate types
features = features.with_columns([
    pl.col("stimulus").cast(pl.Utf8),
    pl.col("index").cast(pl.Int64),  # Match TRT data type
    pl.col("content").cast(pl.Utf8),
    pl.col("word_len").cast(pl.Int16),
    pl.col("freq_zipf").cast(pl.Float64),
    pl.col("dep_dist").cast(pl.Int16),
    pl.col("depth").cast(pl.Int16),
    pl.col("integration_cost").cast(pl.Float64),
    pl.col("dep_label").cast(pl.Utf8),
    pl.col("pos_tag").cast(pl.Utf8),
    pl.col("lemma").cast(pl.Utf8),
    pl.col("sentence_id").cast(pl.Int32),
    pl.col("token_id_sent").cast(pl.Int32),
])

# Save features
FEATURES_PARQUET = PROC / "features_by_word.parquet"
FEATURES_CSV = PROC / "features_by_word.csv"

features.write_parquet(FEATURES_PARQUET)
features.write_csv(FEATURES_CSV)

print(f"Saved features: {features.height} rows -> {FEATURES_PARQUET}")
print(f"Columns: {features.columns}")

Saved features: 9905 rows -> data-clean\processed\features_by_word.parquet
Columns: ['stimulus', 'index', 'content', 'word_len', 'freq_zipf', 'dep_dist', 'depth', 'integration_cost', 'dep_label', 'pos_tag', 'lemma', 'sentence_id', 'token_id_sent']


In [13]:
# Merge with TRT data for modeling
print("Merging features with TRT data...")

trt_with_features = trt.join(
    features, 
    on=["stimulus", "index", "content"], 
    how="left"
)

# Save merged data
TRT_FEATURES_PARQUET = PROC / "trt_with_features.parquet"
TRT_FEATURES_CSV = PROC / "trt_with_features.csv"

trt_with_features.write_parquet(TRT_FEATURES_PARQUET)
trt_with_features.write_csv(TRT_FEATURES_CSV)

print(f"Saved merged data: {trt_with_features.height} rows -> {TRT_FEATURES_PARQUET}")

# Basic statistics
null_counts = trt_with_features.null_count()
print("\nNull counts in merged data:")
for col in ["freq_zipf", "dep_dist", "depth", "integration_cost"]:
    nulls = null_counts.select(pl.col(col)).item()
    total = trt_with_features.height
    print(f"{col}: {nulls}/{total} ({nulls/total*100:.1f}%)")

print(f"\nFinal columns: {trt_with_features.columns}")

Merging features with TRT data...
Saved merged data: 36160 rows -> data-clean\processed\trt_with_features.parquet

Null counts in merged data:
freq_zipf: 939/36160 (2.6%)
dep_dist: 102/36160 (0.3%)
depth: 102/36160 (0.3%)
integration_cost: 102/36160 (0.3%)

Final columns: ['subject_id', 'stimulus', 'condition', 'index', 'content', 'total_reading_time', 'word_len', 'freq_zipf', 'dep_dist', 'depth', 'integration_cost', 'dep_label', 'pos_tag', 'lemma', 'sentence_id', 'token_id_sent']


In [14]:
# Coverage and validation report
print("\n=== ALIGNMENT COVERAGE REPORT ===")
good_coverage = sum(1 for _, cov, _, _ in coverage_stats if cov >= 0.9)
print(f"Stimuli with ≥90% token alignment: {good_coverage}/{len(coverage_stats)}")

print("\nPer-stimulus coverage:")
for stimulus, coverage, total_tokens, aligned_tokens in sorted(coverage_stats, key=lambda x: x[1]):
    print(f"{stimulus}: {coverage:.1%} ({aligned_tokens}/{total_tokens})")

print("\n=== FEATURE SUMMARY ===")
feature_summary = features.select([
    pl.col("word_len").mean().alias("avg_word_len"),
    pl.col("freq_zipf").mean().alias("avg_freq_zipf"),
    pl.col("dep_dist").mean().alias("avg_dep_dist"),
    pl.col("depth").mean().alias("avg_depth"),
    pl.col("integration_cost").mean().alias("avg_integration_cost"),
]).to_pandas().iloc[0]

for col, val in feature_summary.items():
    print(f"{col}: {val:.2f}")

print("\n=== DEPENDENCY RELATIONS ===")
dep_counts = features.group_by("dep_label").len().sort("len", descending=True).head(10)
print(dep_counts.to_pandas())

print("\nPipeline completed successfully!")
print(f"Ready for mixed-effects modeling with {trt_with_features.height} observations")
print(f"Subjects: {trt_with_features['subject_id'].n_unique()}")
print(f"Conditions: {sorted(trt_with_features['condition'].unique().to_list())}")


=== ALIGNMENT COVERAGE REPORT ===
Stimuli with ≥90% token alignment: 152/152

Per-stimulus coverage:
prize-pos.text.0: 95.5% (84/88)
blackout-pos.text.2: 96.3% (79/82)
prize-pos.text.4: 97.0% (32/33)
prize-zero.text.0: 97.3% (108/111)
delayed-pos.text.3: 97.5% (39/40)
blackout-pos.text.1: 97.7% (85/87)
breakfast-pos.text.0: 97.7% (85/87)
breakfast-pos.text.1: 98.3% (113/115)
goldfish-pos.text.0: 98.5% (64/65)
goldfish-pos.text.2: 98.6% (68/69)
prize-pos.text.2: 98.7% (74/75)
prize-pos.text.3: 98.8% (81/82)
breakfast-pos.text.3: 98.8% (84/85)
voicemail-pos.text.1: 98.9% (88/89)
delayed-pos.text.2: 98.9% (88/89)
blackout-pos.text.0: 98.9% (91/92)
delayed-neg.text.0: 99.1% (107/108)
delayed-zero.text.0: 99.1% (107/108)
blackout-zero.text.2: 99.2% (122/123)
prize-neg.text.1: 100.0% (147/147)
blackout-neg.text.2: 100.0% (112/112)
delayed-neg.text.1: 100.0% (105/105)
voicemail-neg.text.1: 100.0% (120/120)
goldfish-pos.text.4: 100.0% (78/78)
delayed-neg.text.4: 100.0% (100/100)
breakfast-zer